In [1]:
!pip install pyspark


In [3]:
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import zipfile

spark = SparkSession.builder.appName("PysparkPractice").getOrCreate()

In [4]:
uploaded = files.upload()

Saving pyspark_logistics_practice_set_2.zip to pyspark_logistics_practice_set_2.zip


In [5]:
zip_name = next(name for name in uploaded if name.endswith(".zip"))

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(".")
    print(z.namelist())

['shipments_75000.csv', 'shipments_extra_12000.csv', 'warehouse_master.csv', 'carrier_master.csv', 'route_master.csv', 'warehouse_sensor_config.json']


In [6]:
shipments_df = spark.read.csv("shipments_75000.csv", header=True, inferSchema=True)
extra_df = spark.read.csv("shipments_extra_12000.csv", header=True, inferSchema=True)
warehouse_df = spark.read.csv("warehouse_master.csv", header=True, inferSchema=True)
carrier_df = spark.read.csv("carrier_master.csv", header=True, inferSchema=True)
route_df = spark.read.csv("route_master.csv", header=True, inferSchema=True)
sensor_df = spark.read.option("multiline", "true").json("warehouse_sensor_config.json")

**PART A**

In [7]:
shipments_df.printSchema()

print("Total records:", shipments_df.count())

shipments_df.show(15)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total records: 75000
+-----------+------------+-------------+--------------

In [8]:
shipments_df.select(
    "shipment_id", "warehouse_id", "service_level", "shipment_status", "shipping_fee_inr"
).show()

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
| SHP0000001|         W08|      Express|      Delivered|          554.34|
| SHP0000002|         W08|     Standard|      Delivered|          414.56|
| SHP0000003|         W01|      Express|      Delivered|          724.64|
| SHP0000004|         W03|     Standard|      Delivered|          287.86|
| SHP0000005|         W03|      Economy|      Delivered|          251.68|
| SHP0000006|         W01|      Express|      Delivered|           477.9|
| SHP0000007|         W02|     Standard|     In Transit|          298.68|
| SHP0000008|         W03|      Economy|     In Transit|          284.52|
| SHP0000009|         W02|     Standard|      Delivered|          861.74|
| SHP0000010|         W02|      Economy|      Delivered|          309.76|
| SHP0000011|         W04|     Standar

In [9]:
express_heavy_df = shipments_df.filter(
    (F.col("service_level") == "Express") & (F.col("package_weight_kg") > 5)
)

express_heavy_df.show()

express_heavy_df.count()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000003|  2026-01-31|   2026-02-01|            2026-02-03| 

4322

In [10]:
shipments_df.orderBy(F.col("shipping_fee_inr").desc()).show(20)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0044904|  2026-08-11|   2026-08-13|            2026-08-16| 

**PART B**

In [11]:
shipments_df.groupBy("shipment_status") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show()

+---------------+-----+
|shipment_status|count|
+---------------+-----+
|      Delivered|52434|
|     In Transit|10472|
|       Returned| 6130|
|      Cancelled| 3723|
|           Lost| 2241|
+---------------+-----+



In [12]:
shipments_df.filter(F.col("shipping_cost_inr").isNotNull()) \
    .groupBy("service_level") \
    .agg(
        F.sum("shipping_fee_inr").alias("total_fees"),
        F.sum("shipping_cost_inr").alias("total_cost")
    ) \
    .withColumn("profit", F.col("total_fees") - F.col("total_cost")) \
    .withColumn("profit_margin_pct", F.col("profit") / F.col("total_fees") * 100) \
    .select(
        "service_level",
        F.round("total_fees", 2).alias("total_shipping_fees"),
        F.round("total_cost", 2).alias("total_shipping_cost"),
        F.round("profit", 2).alias("profit"),
        F.round("profit_margin_pct", 2).alias("profit_margin_pct")
    ) \
    .orderBy("service_level") \
    .show()

+-------------+-------------------+-------------------+----------+-----------------+
|service_level|total_shipping_fees|total_shipping_cost|    profit|profit_margin_pct|
+-------------+-------------------+-------------------+----------+-----------------+
|      Economy|          4404331.7|          3386212.5| 1018119.2|            23.12|
|      Express|         9865578.22|          7350805.7|2514772.52|            25.49|
|     Standard|      1.315425914E7|       1.01104659E7|3043793.24|            23.14|
+-------------+-------------------+-------------------+----------+-----------------+



In [13]:
shipments_df.filter(F.col("shipping_cost_inr").isNotNull()) \
    .groupBy("service_level") \
    .agg(
        F.sum("shipping_fee_inr").alias("total_fees"),
        F.sum("shipping_cost_inr").alias("total_cost")
    ) \
    .withColumn("profit", F.col("total_fees") - F.col("total_cost")) \
    .withColumn("profit_margin_pct", F.col("profit") / F.col("total_fees") * 100) \
    .select(
        "service_level",
        F.round("total_fees", 2).alias("total_shipping_fees"),
        F.round("total_cost", 2).alias("total_shipping_cost"),
        F.round("profit", 2).alias("profit"),
        F.round("profit_margin_pct", 2).alias("profit_margin_pct")
    ) \
    .orderBy("service_level") \
    .show()

+-------------+-------------------+-------------------+----------+-----------------+
|service_level|total_shipping_fees|total_shipping_cost|    profit|profit_margin_pct|
+-------------+-------------------+-------------------+----------+-----------------+
|      Economy|          4404331.7|          3386212.5| 1018119.2|            23.12|
|      Express|         9865578.22|          7350805.7|2514772.52|            25.49|
|     Standard|      1.315425914E7|       1.01104659E7|3043793.24|            23.14|
+-------------+-------------------+-------------------+----------+-----------------+



In [15]:
shipments_df.join(carrier_df, on="carrier_id", how="inner") \
    .join(route_df, on="route_id", how="inner") \
    .filter(F.col("shipping_cost_inr").isNotNull() & (F.col("distance_km") > 0)) \
    .withColumn("cost_per_km", F.col("shipping_cost_inr") / F.col("distance_km")) \
    .groupBy("carrier_id", "carrier_name") \
    .agg(F.round(F.avg("cost_per_km"), 4).alias("avg_cost_per_km")) \
    .orderBy("carrier_id") \
    .show()

+----------+--------------------+---------------+
|carrier_id|        carrier_name|avg_cost_per_km|
+----------+--------------------+---------------+
|       C01|   SwiftLine Express|         0.6033|
|       C02|        ParcelBridge|         0.5957|
|       C03|MetroRoute Logistics|         0.5943|
|       C04|  AirSprint Couriers|         0.6126|
|       C05|           CargoNest|         0.5897|
|       C06|           RouteWise|         0.5879|
|       C07|   BluePeak Delivery|         0.6075|
+----------+--------------------+---------------+



**Part C**

In [16]:
shipments_df.select(
    F.count(F.when(F.col("carrier_id").isNull(), 1)).alias("missing_carrier_id"),
    F.count(F.when(F.col("package_weight_kg").isNull(), 1)).alias("missing_package_weight_kg"),
    F.count(F.when(F.col("shipping_cost_inr").isNull(), 1)).alias("missing_shipping_cost_inr"),
    F.count(F.when(F.col("customer_rating").isNull(), 1)).alias("missing_customer_rating")
).show()

+------------------+-------------------------+-------------------------+-----------------------+
|missing_carrier_id|missing_package_weight_kg|missing_shipping_cost_inr|missing_customer_rating|
+------------------+-------------------------+-------------------------+-----------------------+
|               871|                     1129|                     1503|                  26255|
+------------------+-------------------------+-------------------------+-----------------------+



In [17]:
handling_filled_df = shipments_df.na.fill({"handling_minutes": 0})

handling_filled_df.filter(F.col("handling_minutes").isNull()).count()

0

In [18]:
carrier_known_df = shipments_df.na.drop(subset=["carrier_id"])

carrier_known_df.count()

74129

In [19]:
weight_df = shipments_df.withColumn(
    "weight_category",
    F.when(F.col("package_weight_kg").isNull(), "Unknown")
     .when(F.col("package_weight_kg") < 2, "Light")
     .when(F.col("package_weight_kg") <= 10, "Medium")
     .otherwise("Heavy")
)

weight_df.groupBy("weight_category").count().orderBy(F.col("count").desc()).show()

+---------------+-----+
|weight_category|count|
+---------------+-----+
|         Medium|48577|
|          Light|22670|
|          Heavy| 2624|
|        Unknown| 1129|
+---------------+-----+



**PART D**

In [20]:
dated_df = shipments_df \
    .withColumn("booking_date", F.to_date("booking_date", "yyyy-MM-dd")) \
    .withColumn("dispatch_date", F.to_date("dispatch_date", "yyyy-MM-dd")) \
    .withColumn("promised_delivery_date", F.to_date("promised_delivery_date", "yyyy-MM-dd")) \
    .withColumn("actual_delivery_date", F.to_date("actual_delivery_date", "yyyy-MM-dd"))

dated_df.select("booking_date", "dispatch_date", "promised_delivery_date", "actual_delivery_date").printSchema()

root
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)



In [21]:
delivered_df = dated_df.filter(F.col("shipment_status") == "Delivered") \
    .withColumn("delivery_duration_days", F.datediff("actual_delivery_date", "booking_date")) \
    .withColumn("dispatch_to_delivery_days", F.datediff("actual_delivery_date", "dispatch_date"))

delivered_df.select(
    "shipment_id", "booking_date", "dispatch_date", "actual_delivery_date",
    "delivery_duration_days", "dispatch_to_delivery_days"
).show(10)

delivered_df.agg(
    F.round(F.avg("delivery_duration_days"), 2).alias("avg_days"),
    F.min("delivery_duration_days").alias("min_days"),
    F.max("delivery_duration_days").alias("max_days")
).show()

+-----------+------------+-------------+--------------------+----------------------+-------------------------+
|shipment_id|booking_date|dispatch_date|actual_delivery_date|delivery_duration_days|dispatch_to_delivery_days|
+-----------+------------+-------------+--------------------+----------------------+-------------------------+
| SHP0000001|  2026-05-24|   2026-05-24|          2026-05-27|                     3|                        3|
| SHP0000002|  2026-05-17|   2026-05-18|          2026-05-21|                     4|                        3|
| SHP0000003|  2026-01-31|   2026-02-01|          2026-02-04|                     4|                        3|
| SHP0000004|  2026-03-12|   2026-03-12|          2026-03-18|                     6|                        6|
| SHP0000005|  2026-08-16|   2026-08-16|          2026-08-29|                    13|                       13|
| SHP0000006|  2026-02-06|   2026-02-08|          2026-02-09|                     3|                        1|
|

In [22]:
performance_df = dated_df.withColumn(
    "delivery_performance",
    F.when(F.col("actual_delivery_date").isNull(), "No Delivery Date")
     .when(F.col("actual_delivery_date") < F.col("promised_delivery_date"), "Early")
     .when(F.col("actual_delivery_date") == F.col("promised_delivery_date"), "On Time")
     .otherwise("Late")
)

performance_df.groupBy("delivery_performance").count().orderBy(F.col("count").desc()).show()

+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|    No Delivery Date|22566|
|             On Time|22049|
|                Late|21900|
|               Early| 8485|
+--------------------+-----+



In [23]:
dated_df.filter(F.col("shipment_status") == "Delivered") \
    .withColumn("year_month", F.date_format("booking_date", "yyyy-MM")) \
    .groupBy("year_month") \
    .agg(
        F.count("*").alias("delivered_shipments"),
        F.round(F.sum("shipping_fee_inr"), 2).alias("total_shipping_fees")
    ) \
    .orderBy("year_month") \
    .show()

+----------+-------------------+-------------------+
|year_month|delivered_shipments|total_shipping_fees|
+----------+-------------------+-------------------+
|   2026-01|               6710|         2525138.44|
|   2026-02|               6097|          2268113.5|
|   2026-03|               6810|         2530456.24|
|   2026-04|               6379|         2389163.46|
|   2026-05|               6858|          2549684.0|
|   2026-06|               6615|         2447590.12|
|   2026-07|               6773|         2512355.22|
|   2026-08|               6192|         2306432.52|
+----------+-------------------+-------------------+



**PART E**

In [24]:
import time
import os
from pyspark import StorageLevel


In [25]:
carrier_region_df = shipments_df \
    .join(warehouse_df, on="warehouse_id", how="inner") \
    .join(carrier_df, on="carrier_id", how="inner") \
    .groupBy("region", "carrier_id", "carrier_name") \
    .agg(F.sum("shipping_fee_inr").alias("total_fees"))

fee_window = Window.partitionBy("region").orderBy(F.col("total_fees").desc())

carrier_region_df \
    .withColumn("fee_rank", F.dense_rank().over(fee_window)) \
    .select("region", "carrier_id", "carrier_name", F.round("total_fees", 2).alias("total_fees"), "fee_rank") \
    .orderBy("region", "fee_rank") \
    .show(30, truncate=False)

+------+----------+--------------------+----------+--------+
|region|carrier_id|carrier_name        |total_fees|fee_rank|
+------+----------+--------------------+----------+--------+
|East  |C01       |SwiftLine Express   |393862.62 |1       |
|East  |C02       |ParcelBridge        |373590.02 |2       |
|East  |C03       |MetroRoute Logistics|302260.92 |3       |
|East  |C05       |CargoNest           |243322.92 |4       |
|East  |C07       |BluePeak Delivery   |222025.86 |5       |
|East  |C04       |AirSprint Couriers  |211583.64 |6       |
|East  |C06       |RouteWise           |195810.34 |7       |
|North |C02       |ParcelBridge        |731442.1  |1       |
|North |C01       |SwiftLine Express   |653223.62 |2       |
|North |C03       |MetroRoute Logistics|597883.46 |3       |
|North |C05       |CargoNest           |464601.86 |4       |
|North |C07       |BluePeak Delivery   |425396.7  |5       |
|North |C04       |AirSprint Couriers  |393181.08 |6       |
|North |C06       |Route

In [27]:
running_window = Window.partitionBy("warehouse_id") \
    .orderBy("booking_date", "shipment_id") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_df = shipments_df.withColumn(
    "running_total_fees",
    F.sum("shipping_fee_inr").over(running_window)
)

running_df.filter(F.col("warehouse_id") == "W01") \
    .select("warehouse_id", "booking_date", "shipment_id", "shipping_fee_inr", "running_total_fees") \
    .orderBy("booking_date", "shipment_id") \
    .show(
    )

+------------+------------+-----------+----------------+------------------+
|warehouse_id|booking_date|shipment_id|shipping_fee_inr|running_total_fees|
+------------+------------+-----------+----------------+------------------+
|         W01|  2026-01-01| SHP0004062|          312.48|            312.48|
|         W01|  2026-01-01| SHP0006739|          227.02|             539.5|
|         W01|  2026-01-01| SHP0007553|          316.02|            855.52|
|         W01|  2026-01-01| SHP0008505|          572.68|1428.1999999999998|
|         W01|  2026-01-01| SHP0009674|           322.9|            1751.1|
|         W01|  2026-01-01| SHP0011312|           390.1|            2141.2|
|         W01|  2026-01-01| SHP0011588|          421.72|           2562.92|
|         W01|  2026-01-01| SHP0011698|          190.22|           2753.14|
|         W01|  2026-01-01| SHP0011973|          384.64|3137.7799999999997|
|         W01|  2026-01-01| SHP0017305|          203.48|3341.2599999999998|
|         W0

**PART F**

In [28]:
combined_df = shipments_df.unionByName(extra_df)

combined_df.count()

87000

In [29]:
repeated_ids_df = combined_df.groupBy("shipment_id") \
    .count() \
    .filter(F.col("count") > 1)

repeated_ids_df.show(10)

repeated_ids_df.count()

+-----------+-----+
|shipment_id|count|
+-----------+-----+
| SHP0039848|    2|
| SHP0048411|    2|
| SHP0052790|    2|
| SHP0026413|    2|
| SHP0041611|    2|
| SHP0010090|    2|
| SHP0038340|    2|
| SHP0027530|    2|
| SHP0031755|    2|
| SHP0034259|    2|
+-----------+-----+
only showing top 10 rows


750

In [30]:
deduped_df = combined_df.dropDuplicates(["shipment_id"])

print("Before:", combined_df.count())
print("After:", deduped_df.count())
print("After full-row dropDuplicates:", combined_df.dropDuplicates().count())

Before: 87000
After: 86250
After full-row dropDuplicates: 86250


In [31]:
sensor_df.printSchema()

root
 |-- environment: struct (nullable = true)
 |    |-- humidity_target_pct: long (nullable = true)
 |    |-- temperature_range_c: struct (nullable = true)
 |    |    |-- maximum: long (nullable = true)
 |    |    |-- minimum: long (nullable = true)
 |-- safety: struct (nullable = true)
 |    |-- fire_suppression: string (nullable = true)
 |    |-- inspections: struct (nullable = true)
 |    |    |-- last_date: string (nullable = true)
 |    |    |-- next_date: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- enabled: boolean (nullable = true)
 |    |    |-- sensor_id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- warehouse_id: string (nullable = true)



In [32]:
sensor_flat_df = sensor_df.select(
    "warehouse_id",
    F.col("environment.temperature_range_c.minimum").alias("min_temp_c"),
    F.col("environment.temperature_range_c.maximum").alias("max_temp_c"),
    F.col("environment.humidity_target_pct").alias("humidity_target_pct"),
    F.explode("sensors").alias("sensor")
).select(
    "warehouse_id",
    "min_temp_c",
    "max_temp_c",
    "humidity_target_pct",
    F.col("sensor.sensor_id").alias("sensor_id"),
    F.col("sensor.type").alias("sensor_type"),
    F.col("sensor.enabled").alias("enabled")
)

sensor_flat_df.orderBy("warehouse_id", "sensor_id").show(30)

sensor_flat_df.count()

+------------+----------+----------+-------------------+---------+-----------+-------+
|warehouse_id|min_temp_c|max_temp_c|humidity_target_pct|sensor_id|sensor_type|enabled|
+------------+----------+----------+-------------------+---------+-----------+-------+
|         W01|        18|        32|                 45|   W01-S1|Temperature|  false|
|         W01|        18|        32|                 45|   W01-S2|   Humidity|   true|
|         W01|        18|        32|                 45|   W01-S3|       Door|   true|
|         W02|        18|        32|                 50|   W02-S1|Temperature|   true|
|         W02|        18|        32|                 50|   W02-S2|   Humidity|   true|
|         W02|        18|        32|                 50|   W02-S3|       Door|   true|
|         W03|        18|        32|                 55|   W03-S1|Temperature|   true|
|         W03|        18|        32|                 55|   W03-S2|   Humidity|   true|
|         W03|        18|        32|       

24

**PART G**

In [33]:
pipeline_plan_df = shipments_df \
    .filter((F.col("shipment_status") == "Delivered") & F.col("shipping_cost_inr").isNotNull()) \
    .join(warehouse_df, on="warehouse_id", how="inner") \
    .withColumn("profit", F.col("shipping_fee_inr") - F.col("shipping_cost_inr")) \
    .groupBy("region") \
    .agg(
        F.count("*").alias("shipments"),
        F.round(F.sum("shipping_fee_inr"), 2).alias("total_fees"),
        F.round(F.sum("profit"), 2).alias("total_profit")
    ) \
    .orderBy(F.col("total_profit").desc())

pipeline_plan_df.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_profit#1414 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_profit#1414 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2458]
      +- HashAggregate(keys=[region#94], functions=[count(1), sum(shipping_fee_inr#28), sum(profit#1411)])
         +- Exchange hashpartitioning(region#94, 200), ENSURE_REQUIREMENTS, [plan_id=2455]
            +- HashAggregate(keys=[region#94], functions=[partial_count(1), partial_sum(shipping_fee_inr#28), partial_sum(profit#1411)])
               +- Project [shipping_fee_inr#28, region#94, (shipping_fee_inr#28 - shipping_cost_inr#29) AS profit#1411]
                  +- BroadcastHashJoin [warehouse_id#22], [warehouse_id#91], Inner, BuildRight, false
                     :- Project [warehouse_id#22, shipping_fee_inr#28, shipping_cost_inr#29]
                     :  +- Filter (((isnotnull(shipment_status#30) AND (shipment_status#30 = Delivered)) AND isnotnull(shipping_cos

In [34]:
pipeline_plan_df.show()

+------+---------+----------+------------+
|region|shipments|total_fees|total_profit|
+------+---------+----------+------------+
| South|    22761|8496876.32|  2029173.12|
|  West|    18365|6824399.38|  1618355.08|
| North|     6690|2497038.48|   602906.68|
|  East|     3601|1330736.78|   315148.48|
+------+---------+----------+------------+



In [35]:
delivered_cached_df = shipments_df.filter(F.col("shipment_status") == "Delivered")

delivered_cached_df.cache()

start = time.time()
first_count = delivered_cached_df.count()
first_count_time = time.time() - start

start = time.time()
second_count = delivered_cached_df.count()
second_count_time = time.time() - start

print("First count:", first_count, "-", round(first_count_time, 4), "seconds")
print("Second count:", second_count, "-", round(second_count_time, 4), "seconds")

First count: 52434 - 2.6819 seconds
Second count: 52434 - 0.1154 seconds


In [36]:
start = time.time()
agg_first = delivered_cached_df.groupBy("service_level") \
    .agg(F.round(F.sum("shipping_fee_inr"), 2).alias("total_fees")) \
    .orderBy("service_level") \
    .collect()
agg_first_time = time.time() - start

start = time.time()
agg_second = delivered_cached_df.groupBy("service_level") \
    .agg(F.round(F.sum("shipping_fee_inr"), 2).alias("total_fees")) \
    .orderBy("service_level") \
    .collect()
agg_second_time = time.time() - start

print("First aggregation:", round(agg_first_time, 4), "seconds")
print("Second aggregation:", round(agg_second_time, 4), "seconds")

for row in agg_second:
    print(row["service_level"], row["total_fees"])

First aggregation: 0.943 seconds
Second aggregation: 0.3594 seconds
Economy 3125558.62
Express 7053103.12
Standard 9350271.76


In [37]:
express_df = shipments_df.filter(F.col("service_level") == "Express")

express_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Express shipments:", express_df.count())

express_df.groupBy("shipment_status").count().orderBy(F.col("count").desc()).show()

print(express_df.storageLevel)

Express shipments: 21090
+---------------+-----+
|shipment_status|count|
+---------------+-----+
|      Delivered|14819|
|     In Transit| 2944|
|       Returned| 1767|
|      Cancelled|  985|
|           Lost|  575|
+---------------+-----+

Disk Memory Serialized 1x Replicated


In [38]:
express_df.unpersist(blocking=True)

print("Is cached:", express_df.is_cached)


Is cached: False


**PART H**

In [39]:
shipments_8_df = shipments_df.repartition(8)

print("Partitions:", shipments_8_df.rdd.getNumPartitions())

shipments_8_df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



In [40]:
shipments_12_df = shipments_df.repartition(12)

coalesced_3_df = shipments_12_df.coalesce(3)
repartitioned_3_df = shipments_12_df.repartition(3)

print("Initial:", shipments_12_df.rdd.getNumPartitions())
print("Coalesce(3):", coalesced_3_df.rdd.getNumPartitions())
print("Repartition(3):", repartitioned_3_df.rdd.getNumPartitions())

Initial: 12
Coalesce(3): 3
Repartition(3): 3


In [41]:
print("COALESCE PLAN")
coalesced_3_df.explain()

print("REPARTITION PLAN")
repartitioned_3_df.explain()

COALESCE PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 3
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3187]
            +- FileScan csv [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route_id#24,customer_segment#25,service_level#26,package_weight_kg#27,shipping_fee_inr#28,shipping_cost_inr#29,shipment_status#30,delivery_attempts#31,damage_flag#32,customer_rating#33,handling_minutes#34,payment_mode#35,priority_flag#36] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/shipments_75000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<shipment_id:string,booking_date:date,dispatch_date:date,promised_delivery_date:date,actual...
+- == Initial Plan ==
   Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION

In [42]:
shipment_warehouse_df = shipments_df.join(warehouse_df, on="warehouse_id", how="inner")

shipment_warehouse_df.repartition(4) \
    .write.mode("overwrite") \
    .parquet("shipment_warehouse_parquet")

In [43]:
for name in sorted(os.listdir("shipment_warehouse_parquet")):
    path = os.path.join("shipment_warehouse_parquet", name)
    print(name, "-", os.path.getsize(path), "bytes")

spark.read.parquet("shipment_warehouse_parquet").count()


._SUCCESS.crc - 8 bytes
.part-00000-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet.crc - 3732 bytes
.part-00001-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet.crc - 3736 bytes
.part-00002-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet.crc - 3728 bytes
.part-00003-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet.crc - 3736 bytes
_SUCCESS - 0 bytes
part-00000-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet - 476215 bytes
part-00001-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet - 476783 bytes
part-00002-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet - 476126 bytes
part-00003-2c5b79bc-db5a-45b3-a7a2-3c9b406e3544-c000.snappy.parquet - 476775 bytes


75000

**Final Challenge**

In [44]:
median_weight = shipments_df.approxQuantile("package_weight_kg", [0.5], 0.001)[0]

avg_cost_df = shipments_df.groupBy("service_level") \
    .agg(F.round(F.avg("shipping_cost_inr"), 2).alias("avg_cost_by_service"))

clean_df = shipments_df \
    .join(avg_cost_df, on="service_level", how="left") \
    .withColumn(
        "shipping_cost_clean",
        F.coalesce(F.col("shipping_cost_inr"), F.col("avg_cost_by_service"))
    ) \
    .withColumn(
        "package_weight_clean",
        F.coalesce(F.col("package_weight_kg"), F.lit(median_weight))
    ) \
    .withColumn(
        "carrier_id",
        F.coalesce(F.col("carrier_id"), F.lit("UNASSIGNED"))
    )

print("Median package weight used:", median_weight)

clean_df.select(
    F.count(F.when(F.col("shipping_cost_clean").isNull(), 1)).alias("missing_cost_after"),
    F.count(F.when(F.col("package_weight_clean").isNull(), 1)).alias("missing_weight_after")
).show()

Median package weight used: 2.85
+------------------+--------------------+
|missing_cost_after|missing_weight_after|
+------------------+--------------------+
|                 0|                   0|
+------------------+--------------------+



In [46]:
pipeline_df = clean_df \
    .join(warehouse_df, on="warehouse_id", how="inner") \
    .join(carrier_df, on="carrier_id", how="left") \
    .join(route_df, on="route_id", how="inner") \
    .withColumn("carrier_name", F.coalesce(F.col("carrier_name"), F.lit("Unassigned Carrier"))) \
    .withColumn("booking_date", F.to_date("booking_date", "yyyy-MM-dd")) \
    .withColumn("promised_delivery_date", F.to_date("promised_delivery_date", "yyyy-MM-dd")) \
    .withColumn("actual_delivery_date", F.to_date("actual_delivery_date", "yyyy-MM-dd")) \
    .withColumn("delivery_duration_days", F.datediff("actual_delivery_date", "booking_date")) \
    .withColumn(
        "is_late",
        F.when(F.col("actual_delivery_date") > F.col("promised_delivery_date"), 1).otherwise(0)
    ) \
    .withColumn("shipment_profit", F.col("shipping_fee_inr") - F.col("shipping_cost_clean"))

pipeline_df.select(
    "shipment_id", "region", "carrier_name", "delivery_duration_days",
    "is_late", "shipping_fee_inr", "shipping_cost_clean", "shipment_profit"
).show()

+-----------+------+--------------------+----------------------+-------+----------------+-------------------+-------------------+
|shipment_id|region|        carrier_name|delivery_duration_days|is_late|shipping_fee_inr|shipping_cost_clean|    shipment_profit|
+-----------+------+--------------------+----------------------+-------+----------------+-------------------+-------------------+
| SHP0000001|  West|  AirSprint Couriers|                     3|      1|          554.34|              324.9| 229.44000000000005|
| SHP0000002|  West|        ParcelBridge|                     4|      0|          414.56|              351.6|  62.95999999999998|
| SHP0000003| South|MetroRoute Logistics|                     4|      1|          724.64|              521.4|             203.24|
| SHP0000004|  West|MetroRoute Logistics|                     6|      1|          287.86|              160.1| 127.76000000000002|
| SHP0000005|  West|           CargoNest|                    13|      1|          251.68| 

In [47]:
report_df = pipeline_df.filter(F.col("shipment_status") == "Delivered") \
    .groupBy("region", "carrier_id", "carrier_name") \
    .agg(
        F.count("*").alias("delivered_shipments"),
        F.round(F.sum("shipping_fee_inr"), 2).alias("total_revenue"),
        F.round(F.sum("shipment_profit"), 2).alias("total_profit"),
        F.round(F.avg("delivery_duration_days"), 2).alias("avg_delivery_days"),
        F.round(F.avg("is_late") * 100, 2).alias("late_delivery_pct")
    ) \
    .withColumn("on_time_pct", F.round(100 - F.col("late_delivery_pct"), 2))

rank_window = Window.partitionBy("region").orderBy(F.col("on_time_pct").desc())

final_df = report_df.withColumn("on_time_rank", F.dense_rank().over(rank_window))

In [49]:
final_df.cache()

print("Report rows:", final_df.count())

final_df.orderBy("region", "on_time_rank").show(40)

Report rows: 32
+------+----------+--------------------+-------------------+-------------+------------+-----------------+-----------------+-----------+------------+
|region|carrier_id|        carrier_name|delivered_shipments|total_revenue|total_profit|avg_delivery_days|late_delivery_pct|on_time_pct|on_time_rank|
+------+----------+--------------------+-------------------+-------------+------------+-----------------+-----------------+-----------+------------+
|  East|       C01|   SwiftLine Express|                729|    277425.92|    64875.79|             6.46|            38.41|      61.59|           1|
|  East|       C04|  AirSprint Couriers|                374|    141217.52|    34596.35|             6.54|            40.64|      59.36|           2|
|  East|       C02|        ParcelBridge|                699|     253742.8|    59521.28|             6.62|            41.92|      58.08|           3|
|  East|       C05|           CargoNest|                467|    170922.34|    41153.71|   

In [50]:
final_df.filter(F.col("on_time_rank") == 1) \
    .select("region", "carrier_name", "on_time_pct", "late_delivery_pct") \
    .orderBy("region") \
    .show()

final_df.groupBy("region") \
    .agg(
        F.sum("delivered_shipments").alias("delivered_shipments"),
        F.round(F.sum("total_revenue"), 2).alias("total_revenue"),
        F.round(F.sum("total_profit"), 2).alias("total_profit")
    ) \
    .orderBy("region") \
    .show()

+------+------------------+-----------+-----------------+
|region|      carrier_name|on_time_pct|late_delivery_pct|
+------+------------------+-----------+-----------------+
|  East| SwiftLine Express|      61.59|            38.41|
| North| BluePeak Delivery|      60.96|            39.04|
| South|         CargoNest|      60.44|            39.56|
|  West|Unassigned Carrier|      59.45|            40.55|
+------+------------------+-----------+-----------------+

+------+-------------------+-------------+------------+
|region|delivered_shipments|total_revenue|total_profit|
+------+-------------------+-------------+------------+
|  East|               3690|   1365895.34|   324967.81|
| North|               6800|   2537335.72|   612695.74|
| South|              23214|   8664141.46|  2069490.45|
|  West|              18730|   6961560.98|  1652454.45|
+------+-------------------+-------------+------------+



In [51]:
final_df.write.mode("overwrite") \
    .partitionBy("region") \
    .parquet("logistics_performance_report")

In [52]:
for folder, subfolders, filenames in sorted(os.walk("logistics_performance_report")):
    parts = [f for f in filenames if f.startswith("part-")]
    print(folder, "-", len(parts), "part file(s)")

spark.read.parquet("logistics_performance_report").orderBy("region", "on_time_rank").count()

logistics_performance_report - 0 part file(s)
logistics_performance_report/region=East - 1 part file(s)
logistics_performance_report/region=North - 1 part file(s)
logistics_performance_report/region=South - 1 part file(s)
logistics_performance_report/region=West - 1 part file(s)


32